In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

raw_path = "../data/raw/"
processed_path = "../data/processed/"
figures_path= "../reports/figures/"

In [4]:
# Load dữ liệu từ các file CSV
df_listings = pd.read_csv(os.path.join(raw_path, 'listings.csv'))
df_calendar = pd.read_csv(os.path.join(raw_path, 'calendar_2025.csv'))                  

**PHẦN 1: PHÂN TÍCH SƠ BỘ TẬP DỮ LIỆU CALENDAR_2025**

In [5]:
df_calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   listing_id      495670 non-null  int64  
 1   date            495670 non-null  object 
 2   available       495670 non-null  object 
 3   price           0 non-null       float64
 4   adjusted_price  0 non-null       float64
 5   minimum_nights  495670 non-null  int64  
 6   maximum_nights  495670 non-null  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 26.5+ MB


In [7]:
df_calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,8521,2025-09-28,f,NaN,NaN,2,1125
1,8521,2025-09-29,f,NaN,NaN,2,1125
2,8521,2025-09-30,f,NaN,NaN,2,1125
3,8521,2025-10-01,f,NaN,NaN,2,1125
4,8521,2025-10-02,f,NaN,NaN,2,1125


In [8]:
# Kiểm tra thống kê các cột số (price, minimum_nights, v.v.)
df_calendar.describe()

,listing_id,price,adjusted_price,minimum_nights,maximum_nights
count,4.956700e+05,0.0,0.0,495670.000000,495670.000000
mean,6.303908e+17,NaN,NaN,22.519142,639.369244
std,5.666651e+17,NaN,NaN,36.787509,519.096546
min,8.521000e+03,NaN,NaN,1.000000,1.000000
25%,3.405758e+07,NaN,NaN,2.000000,365.000000
50%,6.827960e+17,NaN,NaN,15.000000,365.000000
75%,1.178492e+18,NaN,NaN,30.000000,1125.000000
max,1.517698e+18,NaN,NaN,700.000000,11250.000000


In [9]:
df_calendar.isnull().sum()  

listing_id             0
date                   0
available              0
price             495670
adjusted_price    495670
minimum_nights         0
maximum_nights         0
dtype: int64

Sau khi khám phá tổng quát về tập dữ liệu, chúng ta thấy được rằng dữ liệu có 7 cột và 495.670 dòng dữ liệu. Điêu đáng chú ý trong tập dữ liệu này có 2 cột bị khuyết thiếu hoàn toàn, đó chính là 'price' và 'adjusted_price' (100% null).Việc giữ lại hai cột này không mang lại bất kỳ giá trị thông tin nào (information gain = 0), đồng thời có thể gây ra ảnh hưởng sai lệch đến phân tích, do đó chúng ta cần phải loại bỏ hoàn toàn hai cột này đi.

In [10]:
df_calendar.dropna(axis=1, how='all', inplace=True)

In [11]:
df_calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   listing_id      495670 non-null  int64 
 1   date            495670 non-null  object
 2   available       495670 non-null  object
 3   minimum_nights  495670 non-null  int64 
 4   maximum_nights  495670 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 18.9+ MB


Sau khi loại bỏ các cột trống hoàn toàn, chúng ta Bước tiếp theo, để đảm bảo tính toàn vẹn của chuỗi thời gian, chúng ta cần tiến hành rà soát các dòng dữ liệu ghi nhận dữ liệu trùng lặp ngày

In [12]:
# Kiểm tra dữ liệu trùng lặp
df_calendar.duplicated().sum()
df_calendar.duplicated(subset=['listing_id', 'date']).sum()

np.int64(0)

Kết quả kiểm tra cho thấy tập dữ liệu Calendar đảm bảo tính vẹn toàn dữ liệu (không có dữ liệu lặp). Dữ liệu đã sạch giá trị nhiễu và sẵn sàng cho bước chuyển đổi đặc trưng. Mô hình học máy không thể tính toán dữ liệu dạng chuỗi của cột available. Do đó, ta cần số hóa cột này: gán giá trị 1 cho những ngày đã được đặt (f) và 0 cho những ngày còn trống (t).

In [13]:
df_calendar['is_booked'] = df_calendar['available'].map({'f': 0, 't': 1})

Sau khi tiến hành binarization, chúng ta tiến hành gom nhóm theo từng căn hộ (listing_id) để tính tỷ lệ lấp đày trung bình trong năm. Hành động này giúp nén dữ liệu thời gian thành bảng dữ liệu chéo, chuẩn bị cho việc hợp nhất dữ liệu với tập listings.

In [14]:
df_occupancy = df_calendar.groupby('listing_id')['is_booked'].mean().reset_index()

In [15]:
df_occupancy.rename(columns={'is_booked': 'occupancy_rate'}, inplace=True)
df_occupancy.head()

,listing_id,occupancy_rate
0,8521,0.257534
1,11169,0.852055
2,19581,0.805479
3,27498,0.934247
4,79762,0.824658


In [16]:
df_occupancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   listing_id      1358 non-null   int64  
 1   occupancy_rate  1358 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 21.3 KB


Sau khi nén dữ liệu từ tập **'calendar_2025'**, chúng ta cần xác nhận xem danh sách các căn hộ này có khớp hoàn toàn với danh sách trong tập **'listings'** hay không. Điều này đảm bảo không có căn hộ nào bị mất mát dữ liệu hoặc xuất hiện các ID lạ trong quá trình xử lý.

In [17]:
calendar_ids = set(df_occupancy['listing_id'])
listing_ids = set(df_listings['id'])

if calendar_ids == listing_ids:
    print("Dữ liệu khớp hoàn toàn 100%")
else:
    print(f"Có {len(calendar_ids - listing_ids)} ID không khớp.")

Dữ liệu khớp hoàn toàn 100%


**Tính toàn vẹn của dữ liệu (Data Integrity)**: Việc đối chiếu cho thấy 100% mã định danh (1.358 id) giữa hai tập dữ liệu đã trùng khớp hoàn toàn. Điều này cho phép chúng ta tiếp tục thực hiện hợp nhất dữ liệu (merge) ở các bước sau mà không lo ngại về dữ liệu rác hoặc thiếu bản ghi.

**Giá trị của biến occupancy_rate**: Thông qua việc gom nhóm (groupby), nhóm đã chuyển đổi gần 500.000 dòng dữ liệu giao dịch theo ngày thành một biến đặc trưng duy nhất cho mỗi căn hộ. Chỉ số này phản ánh **sức hút thị trường** của căn hộ. Thay vì chỉ dựa vào giá niêm yết, tỷ lệ lấp đầy giúp mô hình hiểu được mức độ hiệu quả kinh doanh thực tế, từ đó tối ưu hóa độ chính xác cho mô hình định giá, giúp mô hình tránh trường hợp bẫy **giá ảo**.

**PHẦN 2: KHÁM PHÁ ĐẶC TRƯNG CĂN HỘ (EDA - LISTINGS)**
Sau khi đã xác nhận sự đồng nhất về mã định danh (id) và xây dựng thành công biến số chiến lược occupancy_rate từ dữ liệu thời gian, chúng ta tiến hành chuyển trọng tâm sang tập dữ liệu chính: **listings.csv**.

Tập dữ liệu này đóng vai trò là "khung xương" của mô hình, chứa đựng các thông tin tĩnh về đặc điểm vật lý, vị trí và chính sách của từng căn hộ tại Cambridge. Việc thực hiện EDA (Exploratory Data Analysis) cho tập dữ liệu này nhằm hướng tới 3 mục tiêu trọng tâm:

1. Tinh lọc đặc trưng (Feature Selection): Loại bỏ các trường dữ liệu nhiễu (URL, mô tả văn bản dài, ID người dùng) để tập trung vào các biến có khả năng tác động cao đến giá cả như: room_type, accommodates, bathrooms, bedrooms, và neighbourhood.

2. Làm sạch dữ liệu (Data Cleaning): Xử lý các vấn đề về kiểu dữ liệu (đặc biệt là cột price đang ở dạng chuỗi), xử lý các giá trị thiếu (Missing Values) và phát hiện các điểm dị biệt (Outliers) có thể làm lệch mô hình.

3. Phân tích phân phối: Hiểu rõ bức tranh tổng quan về thị trường Airbnb tại Cambridge (Ví dụ: Khu vực nào tập trung nhiều căn hộ nhất? Mức giá phổ biến là bao nhiêu?).

In [18]:
# kiểm tra tập dữ liệu listings
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1358 non-null   int64  
 1   listing_url                                   1358 non-null   object 
 2   scrape_id                                     1358 non-null   int64  
 3   last_scraped                                  1358 non-null   object 
 4   source                                        1358 non-null   object 
 5   name                                          1358 non-null   object 
 6   description                                   1347 non-null   object 
 7   neighborhood_overview                         694 non-null    object 
 8   picture_url                                   1358 non-null   object 
 9   host_id                                       1358 non-null   i

In [19]:
df_listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,8521,https://www.airbnb.com/rooms/8521,20250928035003,2025-09-28,city scrape,SunsplashedSerenity walk to Harvard & Fresh Pond,"An elegant, sun-splashed, 2 bedroom (+2offices...",Huron Village is known for its charm. We have...,https://a0.muscache.com/pictures/30536/072e0a5...,306681,...,4.92,4.94,4.74,C0121120491,f,2,2,0,0,0.41
1,11169,https://www.airbnb.com/rooms/11169,20250928035003,2025-09-28,city scrape,Lovely Studio Room: Available for long w/ends,Large sunny room w kitchenette & bath. Foam ma...,The neighborhood is quiet and friendly and our...,https://a0.muscache.com/pictures/miso/Hosting-...,40965,...,4.92,4.80,4.77,NaN,f,3,1,2,0,1.01
2,19581,https://www.airbnb.com/rooms/19581,20250928035003,2025-09-28,city scrape,"Furnished suite, Windsor","Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/188f1b4b-f37b...,74249,...,4.91,4.91,4.27,NaN,t,3,0,3,0,0.07
3,27498,https://www.airbnb.com/rooms/27498,20250928035003,2025-09-28,city scrape,Furnished suite 2 @ the Windsor,"Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/bab30c3c-ff3c...,74249,...,4.76,4.88,4.64,NaN,t,3,0,3,0,0.15
4,79762,https://www.airbnb.com/rooms/79762,20250928035003,2025-09-28,city scrape,Cambridge Getaway @ Harvard & MIT,Charming 2-bedroom apartment on the third floo...,Annmarie and I have lived in this area for ove...,https://a0.muscache.com/pictures/airflow/Hosti...,430015,...,4.93,4.93,4.77,STR-15661,f,1,1,0,0,2.59


In [20]:
df_listings.describe()

,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,1.358000e+03,1.358000e+03,1.358000e+03,1358.000000,1358.000000,0.0,1358.000000,1358.000000,1358.000000,1065.000000,...,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1358.000000,1358.000000,1358.000000,1358.000000,1025.000000
mean,6.303908e+17,2.025093e+13,1.777369e+08,698.770250,851.924153,NaN,42.373912,-71.106471,3.270250,1.307981,...,4.744244,4.870020,4.838771,4.872527,4.616556,38.865979,27.310015,10.977172,0.000736,1.525834
std,5.668733e+17,0.000000e+00,1.949916e+08,1681.470771,1939.485396,NaN,0.010625,0.019023,2.315851,0.595237,...,0.417628,0.303552,0.380003,0.281349,0.480144,56.951799,54.644200,20.183577,0.027136,1.909016
min,8.521000e+03,2.025093e+13,3.538400e+04,1.000000,1.000000,NaN,42.353670,-71.157580,1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.414261e+07,2.025093e+13,2.436748e+07,3.000000,4.000000,NaN,42.366240,-71.119623,2.000000,1.000000,...,4.670000,4.860000,4.830000,4.860000,4.530000,2.000000,0.000000,0.000000,0.000000,0.240000
50%,6.827960e+17,2.025093e+13,1.074344e+08,16.000000,24.000000,NaN,42.370599,-71.105840,2.000000,1.000000,...,4.880000,4.950000,4.950000,4.930000,4.720000,8.000000,2.000000,1.000000,0.000000,0.850000
75%,1.178137e+18,2.025093e+13,3.736751e+08,169.000000,234.000000,NaN,42.381308,-71.091340,4.000000,1.500000,...,4.990000,5.000000,5.000000,5.000000,4.860000,83.000000,26.000000,12.000000,0.000000,2.250000
max,1.517698e+18,2.025093e+13,7.207319e+08,5143.000000,5933.000000,NaN,42.400180,-71.071239,16.000000,4.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,169.000000,169.000000,71.000000,1.000000,24.220000


In [21]:
# Xem cột nào đang rỗng nhiều nhất
missing_data = df_listings.isnull().sum().sort_values(ascending=False)
print(missing_data[missing_data > 0])

neighbourhood_group_cleansed    1358
calendar_updated                1358
license                          892
neighbourhood                    664
neighborhood_overview            664
host_about                       480
review_scores_communication      333
last_review                      333
reviews_per_month                333
first_review                     333
review_scores_location           333
review_scores_accuracy           333
review_scores_rating             333
review_scores_value              333
review_scores_cleanliness        333
review_scores_checkin            333
price                            303
estimated_revenue_l365d          303
beds                             300
bathrooms                        293
host_location                    250
host_response_rate               111
host_response_time               111
bedrooms                          93
host_acceptance_rate              86
host_is_superhost                 41
host_neighbourhood                34
h

**ĐÁNH GIÁ CHẤT LƯỢNG BỘ DỮ LIỆU VÀ XÂY DỰNG CHIẾN LƯỢC XỬ LÝ**

Dựa trên kết quả thống kê mô tả, nhóm xác định các vấn đề trọng yếu và đề xuất phương án xử lý như sau:


**1. Kiểm soát biến mục tiêu (price)**: Với ~22% dữ liệu bị khuyết thiếu, nhóm ưu tiên rà soát chéo với tập calendar. Các bản ghi không thể xác định giá niêm yết sẽ bị loại bỏ để bảo vệ độ chính xác của mô hình dự báo.

**2. Tối ưu hóa dữ liệu đặc trưng**: Tận dụng cột bathrooms_text để khôi phục dữ liệu cho biến bathrooms (vốn đang thiếu hụt nghiêm trọng). Đây là kỹ thuật trích xuất đặc trưng (Feature Extraction) từ dữ liệu văn bản.

**3. Lọc nhiễu và Tinh gọn**: Loại bỏ các cột trống 100% và các siêu dữ liệu không hỗ trợ tính toán. Riêng với biến license, nhóm thực hiện mã hóa nhị phân thay vì xóa bỏ để giữ lại thông tin về tính pháp lý.

**4. Chiến lược Imputation (Điền khuyết)**: Sử dụng giá trị trung vị (Median) cho các biến đánh giá (Review Scores) và đặc điểm vật lý (bedrooms, beds) để hạn chế ảnh hưởng của các giá trị cực biên (Outliers).

**THIẾT LẬP QUY TRÌNH XỬ LÝ**

Nhóm triển khai quy trình xử lý theo 5 bước nhằm đảm bảo tính toàn vẹn dữ liệu:


**1. Noise Filtering**: Loại bỏ đặc trưng nhiễu và dữ liệu trống hoàn toàn.

**2. Target Normalization**: Làm sạch và chuẩn hóa biến mục tiêu price.

**3. Numerical Cleaning**: Vệ sinh dữ liệu số, xử lý Missing Values bằng Median và kiểm soát Outliers qua chỉ số $IQR = Q_3 - Q_1$.

**4. Feature Enrichment**: Trích xuất thông tin từ văn bản và mã hóa nhị phân các biến uy tín (license, superhost).

**5. Data Integration**: Hợp nhất đặc trưng căn hộ với chỉ số hiệu suất (occupancy_rate) để hoàn thiện bộ dữ liệu huấn luyện.

Trong tập dữ liệu gốc, trường license chứa các mã định danh pháp lý dưới dạng chuỗi ký tự (Strings). Nhóm thực hiện chuyển đổi thông tin này sang dạng nhị phân (Binary Encoding) để chuẩn bị dữ liệu cho các bước kiểm định sau này:

has_license = 1: Căn hộ có thông tin giấy phép hiện hữu.

has_license = 0: Căn hộ thiếu thông tin hoặc chưa niêm yết giấy phép.

Việc trích xuất này nhằm mục đích giữ lại dữ liệu về trạng thái pháp lý - một yếu tố được kỳ vọng là có tiềm năng tác động đến tâm lý người thuê hoặc chiến lược định giá của chủ nhà. Biến số này sẽ đóng vai trò là một "giả thuyết đầu vào" để nhóm tiến hành phân tích tương quan và kiểm chứng sức ảnh hưởng thực tế ở các giai đoạn sau của dự án.

In [22]:
df_listings['has_license'] = df_listings['license'].notnull().astype(int)

In [23]:
df_listings[['license', 'has_license']].head()

,license,has_license
0,C0121120491,1
1,NaN,0
2,NaN,0
3,NaN,0
4,STR-15661,1


In [24]:
df_listings['host_is_superhost'] = df_listings['host_is_superhost'].map({'t': 1, 'f': 0})

In [25]:
df_listings['host_is_superhost'].head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: host_is_superhost, dtype: float64

In [26]:
df_listings['host_identity_verified'] = df_listings['host_identity_verified'].map({'t': 1, 'f': 0})

In [27]:
df_listings['host_identity_verified'].head()

0    1
1    1
2    1
3    1
4    1
Name: host_identity_verified, dtype: int64

In [28]:
df_listings['has_availability'] = df_listings['has_availability'].map({'t': 1, 'f': 0}) 

In [29]:
df_listings['has_availability'].head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: has_availability, dtype: float64

In [30]:
df_listings['instant_bookable'] = df_listings['instant_bookable'].map({'t': 1, 'f': 0})

In [31]:
df_listings['instant_bookable'].head()

0    0
1    0
2    1
3    1
4    0
Name: instant_bookable, dtype: int64

Nhóm thực hiện loại bỏ các trường thông tin Metadata (URLs, IDs, Scraping info) và các cột trống 100%. Đây là các dữ liệu phát sinh trong quá trình thu thập, không chứa đặc trưng của căn hộ. Cột license gốc cũng chính thức được loại bỏ sau khi đã hoàn tất trích xuất sang biến has_license.

In [32]:
df_listings['price'] = df_listings['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [33]:
df_listings['price']

0        270.0
1        126.0
2        183.0
3        238.0
4        300.0
         ...  
1353     135.0
1354    1166.0
1355    1018.0
1356    1166.0
1357      36.0
Name: price, Length: 1358, dtype: float64

In [36]:
df_listings['price'].describe()

count     1055.000000
mean      1186.094787
std       6211.421226
min         29.000000
25%        105.000000
50%        204.000000
75%        322.000000
max      50031.000000
Name: price, dtype: float64

In [35]:
df_listings.dropna(subset=['price'], inplace=True)

In [40]:
df_listings['bathrooms'].isnull().sum()

np.int64(0)

In [41]:
df_listings['bathrooms_count'] = df_listings['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\baokh\AppData\Local\Temp\ipykernel_23712\1139094750.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_listings['bathrooms_count'] = df_listings['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)


In [42]:
df_listings.loc[df_listings['bathrooms_text'].str.contains('half', case=False, na=False), 'extracted_bath_num'] = 0.5

In [43]:
df_listings['is_shared_bath'] = df_listings['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)

In [57]:
df_listings['is_shared_bath'].head(10)

0     0
1     0
2     0
3     0
4     0
5     0
6     0
9     1
10    0
11    0
Name: is_shared_bath, dtype: int64

In [44]:
df_listings['bathrooms_count'] = df_listings['extracted_bath_num'].fillna(df_listings['bathrooms'])

In [45]:
df_listings['bathrooms_count'] = df_listings['bathrooms_count'].fillna(df_listings['bathrooms_count'].median())

In [46]:
df_listings['bathrooms']

0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
1353    2.0
1354    1.0
1355    1.0
1356    1.0
1357    1.0
Name: bathrooms, Length: 1055, dtype: float64

In [47]:
all_amenities = []
df_listings['amenities'].str.replace(r'["\[\]]', '', regex=True).str.split(',').apply(
    lambda x: [i.strip() for i in x] if isinstance(x, list) else []
).apply(all_amenities.extend)


0       None
1       None
2       None
3       None
4       None
        ... 
1353    None
1354    None
1355    None
1356    None
1357    None
Name: amenities, Length: 1055, dtype: object

In [48]:
from collections import Counter
amenity_counts = Counter(all_amenities)

In [50]:
df_amenity_freq = pd.DataFrame(amenity_counts.items(), columns=['Amenity', 'Count']).sort_values(by='Count', ascending=False)

In [51]:
df_listings['amenities_count'] = df_listings['amenities'].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)

In [52]:
df_listings['amenities_count']

0       46
1       27
2       46
3       50
4       30
        ..
1353    33
1354    32
1355    32
1356    32
1357     7
Name: amenities_count, Length: 1055, dtype: int64

In [53]:
top_amenities = df_amenity_freq.head(20)['Amenity'].tolist()

In [54]:
for amenity in top_amenities:
    col_name = f"has_{amenity.lower().replace(' ', '_')}"
    df_listings[col_name] = df_listings['amenities'].str.contains(amenity, case=False, na=False).astype(int)

In [55]:
print(top_amenities)

['Smoke alarm', 'Carbon monoxide alarm', 'Wifi', 'Hot water', 'Hangers', 'Essentials', 'Bed linens', 'Kitchen', 'Hair dryer', 'Refrigerator', 'Shampoo', 'Iron', 'Microwave', 'Self check-in', 'Dedicated workspace', 'Heating', 'Cooking basics', 'Air conditioning', 'Dishes and silverware', 'TV']


In [342]:
df_listings['bathrooms_text']

0               1 bath
1       1 private bath
2       1 private bath
3       1 private bath
4               1 bath
             ...      
1353           2 baths
1354     1 shared bath
1355     1 shared bath
1356     1 shared bath
1357     1 shared bath
Name: bathrooms_text, Length: 1358, dtype: object

In [345]:
df_listings['bedrooms']

0       2.0
1       1.0
2       1.0
3       1.0
4       2.0
       ... 
1353    2.0
1354    1.0
1355    1.0
1356    1.0
1357    0.0
Name: bedrooms, Length: 1358, dtype: float64

In [298]:
df_listings['bathrooms_count'] = df_listings['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)
df_listings.loc[df_listings['bathrooms_text'].str.contains('half', case=False, na=False), 'bathrooms_count'] = 0.5


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\baokh\AppData\Local\Temp\ipykernel_19224\2164650729.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_listings['bathrooms_count'] = df_listings['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)


In [299]:
df_listings['verifications_count'] = df_listings['host_verifications'].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)

In [300]:
df_listings['verifications_count']

0       2
1       2
2       2
3       2
4       2
       ..
1353    2
1354    2
1355    2
1356    2
1357    1
Name: verifications_count, Length: 1055, dtype: int64

In [301]:
cols_drop = [
    'scrape_id', 'last_scraped', 'source', 'calendar_last_scraped', 'listing_url', 'host_url', 'picture_url', 
    'host_name', 'host_id', 'neighbourhood_group_cleansed', 'calendar_updated',
    'name', 'description', 'neighborhood_overview', 'host_about', 'host_location', 
    'host_verifications', 'host_neighbourhood', 'license', 'bathrooms_text', 'bathrooms', 'amenities', 'host_thumbnail_url', 'host_picture_url'
]

df_listings.drop(columns=cols_drop, inplace=True, errors='ignore')

In [302]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1055 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1015 non-null   object 
 3   host_response_rate                            1015 non-null   object 
 4   host_acceptance_rate                          1023 non-null   object 
 5   host_is_superhost                             1020 non-null   float64
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   object

In [303]:
df_airbnb_cambridge = pd.merge(df_listings, df_occupancy, left_on='id', right_on='listing_id', how='inner')

In [304]:
df_airbnb_cambridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 81 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1015 non-null   object 
 3   host_response_rate                            1015 non-null   object 
 4   host_acceptance_rate                          1023 non-null   object 
 5   host_is_superhost                             1020 non-null   float64
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   o

In [305]:
df_airbnb_cambridge.isnull().sum()

id                            0
host_since                    0
host_response_time           40
host_response_rate           40
host_acceptance_rate         32
                             ..
has_dishes_and_silverware     0
has_tv                        0
verifications_count           0
listing_id                    0
occupancy_rate                0
Length: 81, dtype: int64

In [306]:
cols = ['beds', 'bedrooms', 'bathrooms_count', 'host_response_rate', 'host_acceptance_rate']
for col in cols:
    if df_airbnb_cambridge[col].dtype == 'object':
        df_airbnb_cambridge[col] = df_airbnb_cambridge[col].str.replace('%', '').astype(float)
    df_airbnb_cambridge[col] = df_airbnb_cambridge[col].fillna(df_airbnb_cambridge[col].median())

# Điền 'Unknown' cho các biến phân loại
df_airbnb_cambridge['host_response_time'] = df_airbnb_cambridge['host_response_time'].fillna('Unknown')
df_airbnb_cambridge['host_is_superhost'] = df_airbnb_cambridge['host_is_superhost'].fillna('Unknown')

In [307]:
df_airbnb_cambridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 81 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1055 non-null   object 
 3   host_response_rate                            1055 non-null   float64
 4   host_acceptance_rate                          1055 non-null   float64
 5   host_is_superhost                             1055 non-null   object 
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   o

In [310]:
df_airbnb_cambridge['host_has_profile_pic']

0       t
1       t
2       t
3       t
4       t
       ..
1050    t
1051    t
1052    t
1053    t
1054    t
Name: host_has_profile_pic, Length: 1055, dtype: object

In [311]:
df_airbnb_cambridge['host_identity_verified']

0       t
1       t
2       t
3       t
4       t
       ..
1050    t
1051    t
1052    t
1053    t
1054    t
Name: host_identity_verified, Length: 1055, dtype: object

In [308]:
df_airbnb_cambridge.head(10)


,id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,...,has_self_check-in,has_dedicated_workspace,has_heating,has_cooking_basics,has_air_conditioning,has_dishes_and_silverware,has_tv,verifications_count,listing_id,occupancy_rate
0,8521,2010-12-01,within a few hours,100.0,65.0,1.0,2,3,t,t,...,0,1,1,1,1,1,1,2,8521,0.257534
1,11169,2009-09-24,within a day,100.0,77.0,1.0,4,4,t,t,...,0,1,1,0,1,1,0,2,11169,0.852055
2,19581,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,19581,0.805479
3,27498,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,27498,0.934247
4,79762,2011-03-08,within a few hours,100.0,89.0,1.0,1,2,t,t,...,1,0,1,1,1,1,1,2,79762,0.824658
5,106474,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,106474,0.956164
6,108898,2010-12-01,within a few hours,100.0,65.0,1.0,2,3,t,t,...,0,1,1,1,0,1,1,2,108898,0.046575
7,675441,2012-08-30,Unknown,100.0,96.0,0.0,2,5,t,t,...,0,0,1,0,1,0,1,2,675441,1.000000
8,715532,2012-09-25,within a day,100.0,97.0,1.0,1,3,t,t,...,1,0,1,1,0,1,1,3,715532,0.657534
9,742574,2011-09-13,a few days or more,0.0,0.0,0.0,5,6,t,f,...,1,1,1,1,1,1,1,2,742574,0.326027
